## Preliminary Thoughts
- Our project will be focused on weather conditions during NFL games and creating an index that will display how ideal the current conditions are for playing.
- Weather Conditions that will be focused on:
  - Rain
  - Snow
  - Temperature
  - Wind
  - Wind Chill
  - Heat Index 

In [22]:
from herbie import Herbie, FastHerbie
import pandas as pd, numpy as np
import xarray as xr
import dask
import matplotlib.pyplot as plt
import cartopy.crs as ccrs, cartopy.feature as cfeature
import requests
import pandas as pd
import io
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.colors as mcolors

## Data Download
Ben
- Do not run these cells anymore

In [23]:
# Sets the run time and range for the Herbie run
run = pd.Timestamp("2024-11-18", tz="utc").replace(tzinfo=None).floor('24h')
fxxRange = range(0, 241, 6)

In [24]:
# Herbie run and inventory
FH = FastHerbie([run], model="gfs", product="pgrb2.0p50", fxx=fxxRange, save_dir='/home/bds5658/meteo473/sp26_groupwork/473_sp26_group9/data', overwrite=True)
FH.inventory()

,grib_message,start_byte,end_byte,range,reference_time,valid_time,variable,level,forecast_time,search_this,FILE
0,1,0,294590.0,0-294590,2024-11-18,2024-11-18,PRMSL,mean sea level,anl,:PRMSL:mean sea level:anl,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
1,2,294591,325487.0,294591-325487,2024-11-18,2024-11-18,CLMR,1 hybrid level,anl,:CLMR:1 hybrid level:anl,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
2,3,325488,429328.0,325488-429328,2024-11-18,2024-11-18,ICMR,1 hybrid level,anl,:ICMR:1 hybrid level:anl,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
3,4,429329,508062.0,429329-508062,2024-11-18,2024-11-18,RWMR,1 hybrid level,anl,:RWMR:1 hybrid level:anl,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
4,5,508063,549229.0,508063-549229,2024-11-18,2024-11-18,SNMR,1 hybrid level,anl,:SNMR:1 hybrid level:anl,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
...,...,...,...,...,...,...,...,...,...,...,...
30411,739,161540785,161703342.0,161540785-161703342,2024-11-18,2024-11-28,VGRD,PV=-2e-06 (Km^2/kg/s) surface,240 hour fcst,:VGRD:PV=-2e-06 (Km^2/kg/s) surface:240 hour fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
30412,740,161703343,161866641.0,161703343-161866641,2024-11-18,2024-11-28,TMP,PV=-2e-06 (Km^2/kg/s) surface,240 hour fcst,:TMP:PV=-2e-06 (Km^2/kg/s) surface:240 hour fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
30413,741,161866642,162152692.0,161866642-162152692,2024-11-18,2024-11-28,HGT,PV=-2e-06 (Km^2/kg/s) surface,240 hour fcst,:HGT:PV=-2e-06 (Km^2/kg/s) surface:240 hour fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....
30414,742,162152693,162431126.0,162152693-162431126,2024-11-18,2024-11-28,PRES,PV=-2e-06 (Km^2/kg/s) surface,240 hour fcst,:PRES:PV=-2e-06 (Km^2/kg/s) surface:240 hour fcst,https://noaa-gfs-bdp-pds.s3.amazonaws.com/gfs....


In [25]:
# Defining a search string for the desired variables and downloading the associated GRIB files
variableSearch = ":(TMP:2 m.*|UGRD:10 m.*|VGRD:10 m.*|APCP:surface|CSNOW|CRAIN|PRATE|RH:2 m.*):"
fp = FH.download(variableSearch)

/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  logic = df.search_this.str.contains(search)
/usr/local/anaconda3/envs/custom_envs/meteo473_sp26/lib/python3.13/site-packages/herbie/core.py:978: UserWarning: This pattern is interpreted as a regular expre

In [26]:
# Extracting the data from each of the GRIB files and merging them into one variable using xarray

# The data was broken into three subsets. ds1 is set to a height above ground of 2, specifically for 2m temperature.
ds1 = xr.open_mfdataset(fp, engine = 'cfgrib', backend_kwargs={'filter_by_keys': {'typeOfLevel': 'heightAboveGround', 'level':2}}, combine='nested', concat_dim='valid_time')

# ds2 is set to a level of 10, which was used for gathering 10m wind in the u and v directions.
ds2 = xr.open_mfdataset(fp, engine = 'cfgrib', backend_kwargs={'filter_by_keys': {'typeOfLevel': 'heightAboveGround', 'level':10}}, combine='nested', concat_dim='valid_time')

#ds3 is set to surface level, which is used to monitor the current precipitation at the surface (rain or snow).
ds3 = xr.open_mfdataset(fp, engine = 'cfgrib', backend_kwargs={'filter_by_keys': {'typeOfLevel': 'surface', 'stepType':'instant'}}, combine='nested', concat_dim='valid_time')

ds = xr.merge([ds1, ds2, ds3], compat = 'override')

# Used AI to fix the longitude range to match with cartopy
ds = ds.assign_coords(longitude = (((ds.longitude + 180) % 360) - 180))

Ignoring index file '/home/bds5658/meteo473/sp26_groupwork/473_sp26_group9/data/gfs/20241118/subset_65ef6e41__gfs.t00z.pgrb2.0p50.f000.da267.idx' older than GRIB file
Ignoring index file '/home/bds5658/meteo473/sp26_groupwork/473_sp26_group9/data/gfs/20241118/subset_6512b080__gfs.t00z.pgrb2.0p50.f012.da267.idx' older than GRIB file
Ignoring index file '/home/bds5658/meteo473/sp26_groupwork/473_sp26_group9/data/gfs/20241118/subset_65b2b080__gfs.t00z.pgrb2.0p50.f006.da267.idx' older than GRIB file
Ignoring index file '/home/bds5658/meteo473/sp26_groupwork/473_sp26_group9/data/gfs/20241118/subset_65f1b080__gfs.t00z.pgrb2.0p50.f072.da267.idx' older than GRIB file
Ignoring index file '/home/bds5658/meteo473/sp26_groupwork/473_sp26_group9/data/gfs/20241118/subset_650eb080__gfs.t00z.pgrb2.0p50.f024.da267.idx' older than GRIB file
Ignoring index file '/home/bds5658/meteo473/sp26_groupwork/473_sp26_group9/data/gfs/20241118/subset_650eb080__gfs.t00z.pgrb2.0p50.f036.da267.idx' older than GRIB fil

In [27]:
# Sorting the data chronologically and removing data outside of the continental US. 
ds = ds.sortby('valid_time')
ds = ds.sel(latitude=slice(60,24), longitude=slice(-130, -65))

In [28]:
# Setting the file name and creating the NetCDF file from the data assigned to ds
fname = 'PITCLE'
path = f"/home/bds5658/meteo473/sp26_groupwork/473_sp26_group9/data/{fname}.nc"
ds.to_netcdf(path)

PermissionError: [Errno 13] Permission denied: '/home/bds5658/meteo473/sp26_groupwork/473_sp26_group9/data/PITCLE.nc'

## Ground Truth Observations

Tyler

## Station Model of Cleveland, OH
![Alt text](StationModel.png)

The map gives an overview of what the surface conditions were like all across continguous United States at 00Z on Friday November 22, 2024  (7:00 p.m. EST November 21, 2024), but this is a zoomed in version of that map. The Cleveland, OH station model is indicated by the blue square.
- The station model shows us:
  - Temperatures (indicated by the red number on the top left)
     - Temperature during the game was 34 °F
        - Ignore the red 31 on the bottom right as that shows the temperature from a different station model
  - Dew Point Temperatures (indicated by the green number on the bottom left)
     - Dew Point Temperature was 34 °F
  - Wind Speed (in knots) and Direction (indicated by the line attached to the circle)
     - Wind Speed was 20 knots and the direction coming from the ESE
  - Pressure (last three digits and then add either 9 or 10 before those digits to get the pressure in mb)
     - Unless if there is a hurricane or a generational storm system (neither of which apply here), add 10 if the first digit is anywhere from 0-4
     - Therefore the pressure was 1004.2 mb

## Radar Image during the game
![Alt text](Radar.png)
This map shows the radar in Northern Ohio at 0000 UTC (00Z) on November 22, 2024. Around Cleveland, the radar shows blue and green colors indicating light precipitation during this timeframe. It tells us that there was precipitation falling down during the Steelers vs. Browns game.

## Storms Events Database
![Alt text](StormEventsDatabase.png)

This is a description of what happened during November 21 and 22, 2024 in Cuyahoga County (Cleveland is inside this county). It helps to explain why snowfall occurred and what impacts the area saw. Traffic was less than optimal, so going to and from the game was a hassle for commuters.

## Making the Basemap

Ryan

In [ ]:
# Create the base map
def makebasemap():
    fig = plt.figure(figsize=(24,18))
    ax = plt.axes(projection=ccrs.PlateCarree())
    ax.set_extent([-130, -65, 24, 50])  # CONUS
    ax.add_feature(cfeature.COASTLINE, edgecolor='lightblue')
    ax.add_feature(cfeature.BORDERS)
    ax.add_feature(cfeature.STATES, edgecolor='gray')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue')

#****CODE TO PLOT THE STADIUM NAMES****

# Coordinates for every NFL Stadium
    Stadiums = {
        'State Farm\nStadium': (33.5276, -112.2626),
        'Mercedes-Benz Stadium': (33.7550, -84.4008),
        'M&T Bank Stadium': (39.2779, -76.6227),
        'HighMark Stadium': (42.7738, -78.7868),
        'Bank of America Stadium': (35.2251, -80.8529),
        'Soldier Field': (41.8623, -87.6167),
        'Paycor Stadium': (39.0954, -84.5160),
        'Huntington Bank Field': (41.5061, -81.6995),
        'AT&T Stadium': (32.7473, -97.0945),
        'Empower Field\nat Mile High': (39.7439, -105.0201),
        'Ford Field': (42.3400, -83.0456),
        'Lambeau Field': (44.5013, -88.0622),
        'NRG Stadium': (29.6847, -95.4107),
        'Lucas Oil Stadium': (39.7601, -86.1639),
        'TIAA Bank Field': (30.3239, -81.6373),
        'GEHA Field at\nArrowhead Stadium': (39.0489, -94.4849),
        'Allegiant Stadium': (36.0908, -115.1830),
        'SoFi Stadium': (33.9535, -118.3392),
        'Hard Rock Stadium': (25.9580, -80.2389),
        'U.S. Bank Stadium': (44.9738, -93.2575),
        'Gillette Stadium': (42.0909, -71.2643),
        'Caesars\nSuperdome': (29.9511, -90.0812),
        'Metlife Stadium': (40.8135, -74.0745),
        'Lincoln Financial Field': (39.9008, -75.1675),
        'Acrisure Stadium': (40.4468, -80.0158),
        "Levi's Stadium": (37.4030, -121.9700),
        'Lumen Field': (47.5952, -122.3316),
        'Raymond James Stadium': (27.9759, -82.5033),
        'Nissan Stadium': (36.1665, -86.7713),
        'Northwest Field': (38.9078, -76.8644),
    }

# Plot stadiums
    for stadium, (lat, lon) in Stadiums.items():
        ax.plot(lon, lat, marker='o', color='black', markersize=8,
                markeredgecolor='white', transform=ccrs.PlateCarree())
        ax.text(lon + 0.5, lat + 0.2, stadium, fontsize=8,
                fontweight='bold', transform=ccrs.PlateCarree())
    return fig, ax

## 2m Temperature Map
Ben

In [ ]:
# Setting the file to a variable
fname = 'PITCLE'
path = f"/home/bds5658/meteo473/sp26_groupwork/473_sp26_group9/data/{fname}.nc"
import xarray as xr

fname = 'PITCLE'
path = f"/home/bds5658/meteo473/sp26_groupwork/473_sp26_group9/data/{fname}.nc"

# Open the NetCDF file explicitly with netcdf4 engine
gamedata = xr.open_dataset(path, engine='netcdf4')

# Check it loaded correctly
print(gamedata)
print(list(gamedata.data_vars))

In [ ]:
# Selecting the data for specifically the 96 hour forecast
ts = gamedata.valid_time.values
gd96 = gamedata.sel(valid_time=ts[16])
tempF = (gd96['t2m']- 273.15) * 9/5 + 32

In [ ]:
# Creating a custom colormap for the temperature and real feel maps
# AI was used to figure out how to convert the list of hex codes to a colormap object
tcolors = ['#ff1493','#ff69b4', '#ffb6c1', '#dda0dd', '#9400c3', '#1e90ff', '#40e0d0', '#7cfc00', '#FFFF00', '#ff8500', '#ff0000', '#b22222', '#800000']
tmap = LinearSegmentedColormap.from_list('temperature', tcolors)

In [ ]:
# Creating the map of 2m Temperature
fig, ax = makebasemap()
contt2m = ax.contourf(gd96['longitude'], gd96['latitude'], tempF, cmap = tmap, vmin = -40, vmax = 120, levels = np.arange(-40, 120, .5), transform=ccrs.PlateCarree())
cbart2m = plt.colorbar(contt2m, ax=ax, orientation = 'horizontal', pad=.04)
cbart2m.set_label(label='Temperature (°F)',fontsize=20)
cbart2m.ax.tick_params(labelsize=16)
plt.title("2m Temperature over NFL Stadiums\nInitialized: 2024-11-18 0Z   Valid: 2024-11-22 0Z", fontsize=20)
plt.show()

## Apparent Temperature Map
Ben

In [ ]:
# Grabbing wind speed from data and converting to mph
windms = np.sqrt(gd96['u10']**2+gd96['v10']**2)
windmph=windms*2.237

# Calculating wind chill 
wc = (35.7 + 0.6215 * tempF - 35.75 * (windmph ** 0.16) + 0.4275 * tempF * (windmph ** 0.16))

# Only applying wind chill to areas with temperatures uner 50F and wind > 3 mph
# AI was used to discover the xr.where() function, and format was replicated in the next cell
wctemp = xr.where((tempF <= 50) & (windmph >= 3), wc, tempF)

In [ ]:
# Pulling relative humidity data and calculating heat index from NWS formula
rh = gd96['r2']
hi = (-42.379 + 2.04901523 * tempF + 10.14333127 * rh - 0.22475541 * tempF * rh - 0.00683783 * tempF**2 - 0.05481717 * rh**2 + 0.00122874 * tempF**2 * rh +  0.00085282 * tempF * rh**2 - 0.00000199 * tempF**2 * rh**2)

# NWS formula contains adjustments under certain conditions; these were applied accordingly
# Adjustment for when T is between 80F and 112F and RH < 13%
adj1 = ((13 - rh) / 4) * np.sqrt(np.maximum(0, (17 - np.abs(tempF - 95.)) / 17))
hi = xr.where((rh < 13) & (tempF >= 80) & (tempF <= 112), hi - adj1, hi)

# Adjustment for when T is between 80F and 87F, and RH > 85%
adj2 = ((rh - 85) / 10) * ((87 - tempF) / 5)
hi = xr.where((rh > 85) & (tempF >= 80) & (tempF <= 87), hi + adj2, hi)

# Combining original heat index and both adjustments into one group
hitemp = xr.where(tempF >= 80, hi, tempF)

In [ ]:
# Applying both wind chill and heat index to create a map of real feel
fl = xr.where(tempF <= 50, wctemp, xr.where(tempF >= 80, hitemp, tempF))

In [ ]:
# Creating the map of apparent temperature
fig, ax = makebasemap()
contfl = ax.contourf(gd96['longitude'], gd96['latitude'], fl, cmap=tmap, levels=np.arange(-40, 120, .5), transform=ccrs.PlateCarree())
cbarfl = plt.colorbar(contfl, ax=ax, orientation = 'horizontal', pad=.04)
cbarfl.set_label(label='Apparent Temperature (°F)',fontsize=20)
cbarfl.ax.tick_params(labelsize=16)
plt.title("Apparent Temperature over NFL Stadiums\nInitialized: 2024-11-18 0Z   Valid: 2024-11-22 0Z", fontsize=20)
plt.show()

## Wind Speed and Direction Map

Tyler

In [ ]:
import pandas as pd, numpy as np

initial_datetime = pd.to_datetime(gamedata.valid_time.values[0]) # Creates a variable for the initial datetime of the data
initial_time = initial_datetime.strftime("%Hz %a %b %d %Y") # Converts the initial datetime into a format of time in UTC (Ex. 00Z), abbreviated day of the week (Ex. Mon), abbreviated month (Ex. Nov), day number of the month (Ex. 18), and year (Ex. 2024)
valid_datetime = pd.to_datetime(gamedata.valid_time.values[16]) # Creates a variable for the valid datetime of the data
valid_time = valid_datetime.strftime("%Hz %a %b %d %Y") # Converts the initial datetime into a format of time in UTC, abbreviated day of the week, abbreviated month, day number of the month, and year

In [ ]:
#Create variables for the u component and v component of the wind directions
u10 = gd96["u10"]*1.94384 # Converts m/s to kts
v10 = gd96["v10"]*1.94384

# Convert wind from u and v component vectors into a magnitude 
wind_spd = np.sqrt(u10**2 + v10**2)


fig, ax = makebasemap() # Creates a map
wind_spd_plot = ax.contourf(gd96["longitude"], gd96["latitude"], wind_spd, levels=np.arange(0, 75, 5), cmap='cool', transform=ccrs.PlateCarree()) # Plots the colormap for wind speed intervals of 5 kts from 0 kts to 70 kts
plt.colorbar(wind_spd_plot, ax = ax, orientation = "horizontal", pad = 0.05) # Adds a colorbar below the map to label the wind speed values for each color

# Add wind barbs to the map and reduce the total amount of wind barbs shown
# I used AI to figure out how to reduce the amount because the regrid_shape function didn't work
skip = 4
ax.barbs(gd96["longitude"][::skip], gd96["latitude"][::skip], u10[::skip, ::skip], v10[::skip, ::skip], linewidth=1, length=7, transform=ccrs.PlateCarree())

# Add title with imported initial and valid times
plt.title(f"GFS Wind Speed (kt) and Direction over NFL Stadiums\nInitialized: {initial_time}   Valid: {valid_time}", fontsize=20)
plt.show()

## Precipitation Map

Ryan 

In [ ]:
#CREATE PRECIPITATION MAP 
# Create the base map
fig = plt.figure(figsize=(24,18))
ax = plt.axes(projection=ccrs.PlateCarree())
ax.set_extent([-130, -65, 24, 50])  # CONUS
ax.add_feature(cfeature.COASTLINE, edgecolor='lightblue')
ax.add_feature(cfeature.BORDERS)
ax.add_feature(cfeature.STATES, edgecolor='gray')
ax.add_feature(cfeature.OCEAN, facecolor='lightblue')


#Code to load the precipitation from the GFS model 
# Setting the file to a variable
fname = 'PITCLE'
path = f"/home/bds5658/meteo473/sp26_groupwork/473_sp26_group9/data/{fname}.nc"
gamedata = xr.open_dataset(path)
gamedata

#Change the longitude range 
gamedata = gamedata.assign_coords( longitude = (((gamedata.longitude + 180) % 360) - 180))

# Rain Rate 'RPRATE', snow 'SPRATE'

#Assign variables to the right forecast hour (16)
rain = gamedata['prate'].isel(valid_time=16) * 141.73
snow = gamedata['prate'].isel(valid_time=16) * 141.73 * 10
lat = gamedata['latitude']
lon = gamedata['longitude']

crain = gamedata['crain'].isel(valid_time=16)
csnow = gamedata['csnow'].isel(valid_time=16)

#Mask the zero values 
rain_mask = rain.where(crain >0)
snow_mask = snow.where(csnow >0)

# Coordinates for every NFL Stadium
Stadiums = {
        'State Farm\nStadium': (33.5276, -112.2626),
        'Mercedes-Benz Stadium': (33.7550, -84.4008),
        'M&T Bank Stadium': (39.2779, -76.6227),
        'HighMark Stadium': (42.7738, -78.7868),
        'Bank of America Stadium': (35.2251, -80.8529),
        'Soldier Field': (41.8623, -87.6167),
        'Paycor Stadium': (39.0954, -84.5160),
        'Huntington Bank Field': (41.5061, -81.6995),
        'AT&T Stadium': (32.7473, -97.0945),
        'Empower Field\nat Mile High': (39.7439, -105.0201),
        'Ford Field': (42.3400, -83.0456),
        'Lambeau Field': (44.5013, -88.0622),
        'NRG Stadium': (29.6847, -95.4107),
        'Lucas Oil Stadium': (39.7601, -86.1639),
        'TIAA Bank Field': (30.3239, -81.6373),
        'GEHA Field at\nArrowhead Stadium': (39.0489, -94.4849),
        'Allegiant Stadium': (36.0908, -115.1830),
        'SoFi Stadium': (33.9535, -118.3392),
        'Hard Rock Stadium': (25.9580, -80.2389),
        'U.S. Bank Stadium': (44.9738, -93.2575),
        'Gillette Stadium': (42.0909, -71.2643),
        'Caesars\nSuperdome': (29.9511, -90.0812),
        'Metlife Stadium': (40.8135, -74.0745),
        'Lincoln Financial Field': (39.9008, -75.1675),
        'Acrisure Stadium': (40.4468, -80.0158),
        "Levi's Stadium": (37.4030, -121.9700),
        'Lumen Field': (47.5952, -122.3316),
        'Raymond James Stadium': (27.9759, -82.5033),
        'Nissan Stadium': (36.1665, -86.7713),
        'Northwest Field': (38.9078, -76.8644),
    }

#Rain Intensity color map (got this from AI)
colors = [
    "#00FF00",  # light green
    "#66FF00",
    "#CCFF00",
    "#FFFF00",  # yellow
    "#FFCC00",
    "#FF9900",  # orange
    "#FF4D00",
    "#FF0000",  # red
    "#CC00CC",
    "#8000FF"   # purple
]
rain_cmap = mcolors.LinearSegmentedColormap.from_list("rain_radar", colors)


#Plot the rain onto the map 
rain_plot = ax.pcolormesh(lon, lat, rain_mask, cmap=rain_cmap, shading='auto', alpha=0.55, vmin=0, vmax=0.5, transform=ccrs.PlateCarree())

#Plot the snow onto the map
snow_plot = ax.pcolormesh(lon, lat, snow_mask, cmap='Blues', shading='auto', alpha=0.55, vmin=0, vmax=1, transform=ccrs.PlateCarree())

#Colorbars
cbar_snow = fig.colorbar(snow_plot, ax=ax, orientation='horizontal', pad=0.08, shrink=0.7)
cbar_snow.set_label('Snow Intensity (in/hr)')
cbar_rain = fig.colorbar(rain_plot, ax=ax, orientation='horizontal', pad=0.2, shrink=0.7)
cbar_rain.set_label('Rain Intensity (in/hr)')

# Plot stadiums last so they show up on top of the precip
for stadium, (lat, lon) in Stadiums.items():
    ax.plot(lon, lat, marker='o', color='black', markersize=8,
            markeredgecolor='white', transform=ccrs.PlateCarree())
    ax.text(lon + 0.5, lat + 0.2, stadium, fontsize=8,
            fontweight='bold', transform=ccrs.PlateCarree())

# Add title
plt.title("NFL Stadiums Precipitation Map", fontsize=20)

#Show the final map
plt.show()

## Ideas on Designing our Index 

Tyler and Ryan

- We want to design an index to consider how good or bad the weather is for an NFL game
  - Lower values will indicate good weather for a game
  - Higher values will indicate bad weather
- We will have three separate categories on a scale from 0 to 8:
  - Apparent Temperature 
  - Wind Speed
  - Precipitation
    - 0 to 4 for rain, 0 to 4 for snow
- The values for each of the three categories will be combined into a total value up to 24
  - The closer the value is to 0, the better the conditions will be for an NFL game
  - The closer the value is to 24, the worse the conditions will be for an NFL game
- The index will be useful for NFL players to prepare for their game, as well as spectators that need to know what the conditions will be like before heading into a stadium


## Index Map

Ryan

In [ ]:
# Code for the numerical scale that will be used to determine how bad/good the weather is for a NFL football game 
# This system will work where the 'worse' weather adds points to the scale 


#Code to fix the issue regarding different coordinate systems in precipitation and temperature/wind 
#Code FOR WEATHER HARSHNESS SCALE (USED AI TO HELP WITH CODE TO CREATE THE SCALE AS IT WAS BEYOND JUST IF ELSE STATEMENTS)
# APPARENT TEMPERATURE
temp_points = xr.zeros_like(fl)

temp_points = xr.where((fl >= 75) & (fl < 80), 1, temp_points)
temp_points = xr.where((fl >= 80) & (fl < 85), 2, temp_points)
temp_points = xr.where((fl >= 85) & (fl < 90), 3, temp_points)
temp_points = xr.where((fl >= 90) & (fl < 93), 4, temp_points)
temp_points = xr.where((fl >= 93) & (fl < 96), 5, temp_points)
temp_points = xr.where((fl >= 96) & (fl < 99), 6, temp_points)
temp_points = xr.where((fl >= 100) & (fl < 105), 7, temp_points)
temp_points = xr.where(fl >= 105, 8, temp_points)
temp_points = xr.where((fl >= 50) & (fl < 75), 0, temp_points)
temp_points = xr.where((fl >= 40) & (fl < 50), 1, temp_points)
temp_points = xr.where((fl >= 32) & (fl < 40), 2, temp_points)
temp_points = xr.where((fl >= 27) & (fl < 32), 3, temp_points)
temp_points = xr.where((fl >= 22) & (fl < 27), 4, temp_points)
temp_points = xr.where((fl >= 17) & (fl < 22), 5, temp_points)
temp_points = xr.where((fl >= 10) & (fl < 17), 6, temp_points)
temp_points = xr.where((fl >= 5) & (fl < 10), 7, temp_points)
temp_points = xr.where(fl < 5, 8, temp_points)

#WIND SPEED
wind_points = xr.zeros_like(windmph)
wind_points = xr.where((windmph >= 5) & (windmph < 10), 1, wind_points)
wind_points = xr.where((windmph >= 10) & (windmph < 15), 2, wind_points)
wind_points = xr.where((windmph >= 15) & (windmph < 20), 3, wind_points)
wind_points = xr.where((windmph >= 20) & (windmph < 25), 4, wind_points)
wind_points = xr.where((windmph >= 25) & (windmph < 30), 5, wind_points)
wind_points = xr.where((windmph >= 30) & (windmph < 40), 6, wind_points)
wind_points = xr.where((windmph >= 40) & (windmph < 50), 7, wind_points)
wind_points = xr.where(windmph >= 50, 8, wind_points)

#PRECIPITATION 
rain_points = xr.zeros_like(rain)
rain_points = xr.where((rain > 0) & (rain <= 0.05), 1, rain_points)  
rain_points = xr.where((rain > 0.05) & (rain <= 0.1), 2, rain_points) 
rain_points = xr.where((rain > 0.1) & (rain <= 0.2), 3, rain_points) 
rain_points = xr.where(rain > 0.2, 4, rain_points)                      

# --- Assign points for snow ---
snow_points = xr.zeros_like(snow)
snow_points = xr.where((snow > 0) & (snow <= 0.05), 1, snow_points)  
snow_points = xr.where((snow > 0.05) & (snow <= 0.1), 2, snow_points) 
snow_points = xr.where((snow > 0.1) & (snow <= 0.2), 3, snow_points)  
snow_points = xr.where(snow > 0.2, 4, snow_points)                       





#Create map and colorbar
points = temp_points + wind_points + rain_points + snow_points
fig, ax = makebasemap()
cont = ax.contourf( gd96['longitude'], gd96['latitude'], points, levels=np.arange(-0.5, 25, 1), cmap=rain_cmap, transform=ccrs.PlateCarree())
cbar = plt.colorbar(cont, ax=ax, orientation='horizontal', pad=0.04)
cbar.set_label("Harshness Index", fontsize=14)
cbar.set_ticks(range(0, 25))
plt.title("Weather Harshness Index Map", fontsize=18)
plt.show()
